# Unpacking & **kwargs — Session 03: Named Extras and Star Openers

Where `*args` packs anonymous positional values, `**kwargs` packs **named** extras into a dictionary. The star is symmetric: at a function definition it collects, and at a call site it expands. This notebook is the practical companion to the `*args` theory, focused on real use cases with a products catalogue.

**The big picture.** Optional, named configuration is everywhere: `discount=`, `gst=`, `delivery=`, `gift_wrap=`. `**kwargs` lets a function accept an open-ended set of such options and handle the ones it knows while tolerating the rest. Call-site unpacking does the reverse: an existing list or dictionary is spread into separate arguments with `*` and `**`.

**Why it matters.** Flexible signatures are how libraries evolve without breaking callers. A public function with a `**kwargs` slot can absorb tomorrow's options today. Unpacking turns stored data into calls, which is essential for generic wrappers, `functools.partial`, and configuration-driven code.

**Roadmap.** In this notebook you will:
1. Collect named extras with `**kwargs` and read them safely with `.get`.
2. Combine every parameter kind in one full signature.
3. Unpack lists and dictionaries at the call site.
4. Count packed arguments in a helper.
5. Meet `functools.partial` as a way to freeze arguments.
6. Apply discounting and receipt printing to a real catalogue.

**Key definitions.** *`**kwargs`*: a dictionary of extra keyword arguments. *Unpacking*: `f(*items)` and `f(**mapping)` at the call site. *`partial`*: a `functools` helper that pre-fills some arguments of a function.

## 1. **kwargs — The Tagged Gift-Bag

Placing `**kwargs` in a signature collects any extra *named* arguments into a dictionary called `kwargs`. The double star is the keyword counterpart of `*args`: one packs positionals into a tuple, the other packs keywords into a dict.

```text
def profile(**info):
    ...
profile(name='Sana', city='Delhi', hobby='chess')
# info -> {'name': 'Sana', 'city': 'Delhi', 'hobby': 'chess'}
```

Because the result is an ordinary dictionary, use `.get(key, default)` to read optional fields gracefully — a missing key returns the default instead of raising `KeyError`. An empty call is perfectly valid and yields an empty dictionary.

The bag carries *tags*, so order never matters; each value travels with its name. That is what makes `**kwargs` ideal for optional configuration.

In [1]:
# 🐣 **kwargs — named extras ka dict-pack! 🎁
def profile(**info):
    print(f"type: {type(info).__name__} | keys: {sorted(info)}")
    print(f"  name: {info.get('name', 'N/A')}")
    print(f"  city: {info.get('city', 'unknown')}")

profile(name="Sana", city="Delhi", hobby="chess")
print()
profile()                                            # empty-gift-bag OK!
# Expected OUTPUT:
# type: dict | keys: ['city', 'hobby', 'name']
#   name: Sana
#   city: Delhi
#
# type: dict | keys: []
#   name: N/A
#   city: unknown

type: dict | keys: ['city', 'hobby', 'name']
  name: Sana
  city: Delhi

type: dict | keys: []
  name: N/A
  city: unknown


## 2. The Full Signature

A single signature can combine every parameter kind: `def show(order_id, *items, urgent=False, **meta)`. Reading it left to right, `order_id` is a required positional, `items` is a tuple of any extra positionals, `urgent` is keyword-only, and `meta` absorbs all remaining named arguments.

At the call `show(101, 'book', 'pen', urgent=True, gift='yes')`, the pieces fall into place: `order_id=101`, `items=('book','pen')`, `urgent=True`, and `meta={'gift': 'yes'}`. The order laws from the theory notebook guarantee an unambiguous split.

This pattern is the backbone of flexible APIs: a small set of required values, a variadic payload, a few explicit flags, and an open bag for the rest.

In [2]:
# 🐥 FULL-SIGNATURE combo — sab ek saath! (pos → *args → kw-only → **kwargs)
def show(order_id, *items, urgent=False, **meta):
    print(f"order#{order_id} items={items}")
    print(f"  urgent={urgent} | meta={meta}")

show(101, "book", "pen", urgent=True, gift="yes", note="call-back")
# Expected OUTPUT:
# order#101 items=('book', 'pen')
#   urgent=True | meta={'gift': 'yes', 'note': 'call-back'}

order#101 items=('book', 'pen')
  urgent=True | meta={'gift': 'yes', 'note': 'call-back'}


## 3. Call-Side Unpacking

The star works in both directions. At a call site, `*iterable` spreads a sequence into positional arguments and `**mapping` spreads a dictionary into keyword arguments.

- `add3(*[10, 20, 30])` is equivalent to `add3(10, 20, 30)`.
- `welcome(**{'name': 'Ravi', 'city': 'Pune'})` matches arguments by name, so dictionary order is irrelevant.

This is how stored data becomes a function call. `f(*list)` requires the list length to match the parameters, while `f(**dict)` requires the keys to name existing parameters, or a `**kwargs` slot to catch the rest. Mismatches raise `TypeError`.

In [3]:
# 🐥 Call-side UNPACKING — star se dabba kholo ⭐ (list → args!)
def add3(a, b, c):
    return a + b + c

nums = [10, 20, 30]
print("add3(*nums)     :", add3(*nums), "← list ko * se KHOLA — 3 args ban!")

def welcome(name, city):
    print(f"Welcome {name} from {city}!")

student = {"city": "Pune", "name": "Ravi"}          # order different — dict matches BY NAME!
welcome(**student)                                   # ** se named-arg bomb!
# Expected OUTPUT:
# add3(*nums)     : 60 ← list ko * se KHOLA — 3 args ban!
# Welcome Ravi from Pune!

add3(*nums)     : 60 ← list ko * se KHOLA — 3 args ban!
Welcome Ravi from Pune!


## 4. Counting the Bags

A helper can accept both packs at once: `def packer(*args, **kwargs)`. Inside, `len(args)` counts the anonymous values and `len(kwargs)` counts the tagged ones. Calling it with nothing returns `anonymous=0 tagged=0`, which is a valid and common case.

This is a useful debugging and proxy pattern: a wrapper can report or log how many arguments it received before forwarding them onward. Because both packs are ordinary containers, inspecting them is easy.

In [4]:
# 🐔 kwargs-length check + kw-detective helpers
def packer(*args, **kwargs):
    return f"anonymous={len(args)} tagged={len(kwargs)}"

print(packer(1, 2, 3, x=1, y=2))
print(packer())                                      # zero-flexibility legit!
# Expected OUTPUT:
# anonymous=3 tagged=2
# anonymous=0 tagged=0

anonymous=3 tagged=2
anonymous=0 tagged=0


## 5. functools.partial — Freezing Arguments

`functools.partial` creates a new callable with some arguments already fixed. `eighteen_off = partial(discount_calc, 0.18)` freezes the discount rate, leaving a function that takes only a price. Calling `eighteen_off(48000)` gives the same result as `discount_calc(0.18, 48000)`.

This is a light form of currying: pre-supply the parts that rarely change, and expose a simpler callable for the parts that do. It is handy for callbacks, mapping a transformation over data, and building configured helpers without writing a wrapper function.

In [5]:
# 🐔 partial teaser — curry-ka-chhota-bhai 🍛 (Section 04 OOP ke side-serves!)
from functools import partial

def discount_calc(rate, price):
    return round(price * (1 - rate), 2)

eighteen_off = partial(discount_calc, 0.18)          # rate FIXED, price open!
print("partial      :", eighteen_off(48000), "← Laptop ka 18%-off price")
print("original fnc :", discount_calc(0.18, 48000), "(same result — freeze-then-use convenience!)")
# Expected OUTPUT:
# partial      : 39360.0 ← Laptop ka 18%-off price
# original fnc : 39360.0 (same result — freeze-then-use convenience!)

partial      : 39360.0 ← Laptop ka 18%-off price
original fnc : 39360.0 (same result — freeze-then-use convenience!)


## 6. Real Dataset — The Discount Lab

This section loads `products_catalog.csv` into a name-to-price dictionary and applies a variadic discount function. `discounted_list(rate, *prices)` accepts a rate followed by any number of prices and returns the discounted values.

Calling it as `discounted_list(0.18, *first3)` unpacks the first three prices into positional arguments, demonstrating definition-side packing and call-side unpacking together. Zipping the names with the results produces a small, readable report — the pattern used throughout data tooling.

In [6]:
# 💾 REAL DATASET — Products ka Flexible Discount-Lab! 🏷️
import csv

def load_products():
    for path in ["../../00_Resources/datasets/products_catalog.csv",
                 "00_Python/00_Resources/datasets/products_catalog.csv"]:
        try:
            with open(path, newline="", encoding="utf-8") as f:
                return list(csv.DictReader(f))
        except FileNotFoundError:
            continue
    raise FileNotFoundError("products_catalog.csv nahi mila!")

products = load_products()
name2price = {p["product_name"]: float(p["price"]) for p in products}
print("catalog:", len(name2price), "products ready 🏪")

def discounted_list(rate, *prices):
    return [round(p * (1 - rate), 2) for p in prices]

first3 = list(name2price.values())[:3]
print("\n18%-off first-3 :", discounted_list(0.18, *first3))
print("names zipped    :", list(zip(list(name2price)[:3], discounted_list(0.18, *first3))))
# Expected OUTPUT:
# catalog: 30 products ready 🏪
#
# 18%-off first-3 : [39360.0, 12300.0, 1640.0]
# names zipped    : [('Laptop', 39360.0), ('Smartphone', 12300.0), ('Headphones', 1640.0)]

FileNotFoundError: products_catalog.csv nahi mila!

## 7. The Receipt Printer

`receipt(customer, *items, discount=0.0, gst=0.0, delivery=0.0, **flags)` is the full signature in a realistic setting. It sums item prices from the catalogue, applies discount, then GST, then delivery, and finally prints the remaining unknown flags.

Every parameter kind does a job: `customer` is required, `items` is variadic, the money values are keyword-only with defaults, and `flags` absorbs options such as `gift_wrap='yes'`. The example bill shows the arithmetic step by step, and the flags line proves the open bag is still available for future options.

In [7]:
# 💾 Receipt printer — **kwargs style optional-fields ka showcase! 🧾
def receipt(customer, *items, discount=0.0, gst=0.0, delivery=0.0, **flags):
    subtotal = sum(name2price[i] for i in items)
    after_disc = subtotal * (1 - discount)
    after_gst = after_disc * (1 + gst)
    final = after_gst + delivery
    print(f"  🧾 {customer}'s cart: {len(items)} items")
    for i in items:
        print(f"    - {i}: ₹{name2price[i]:,.0f}")
    print(f"    subtotal ₹{subtotal:,.0f} | -{int(discount*100)}% → ₹{after_disc:,.0f}")
    print(f"    +{int(gst*100)}% GST → ₹{after_gst:,.0f} | +delivery ₹{delivery:,.0f}")
    print(f"    FINAL: ₹{final:,.0f} | flags: {sorted(flags)}")

receipt("Rahul", "Laptop", "Smartphone", "Headphones",
        discount=0.10, gst=0.05, delivery=500, gift_wrap="yes", festival="diwali")
# Expected OUTPUT:
#   🧾 Rahul's cart: 3 items
#     - Laptop: ₹48,000
#     - Smartphone: ₹15,000
#     - Headphones: ₹2,000
#     subtotal ₹65,000 | -10% → ₹58,500
#     +5% GST → ₹61,425 | +delivery ₹500
#     FINAL: ₹61,925 | flags: ['festival', 'gift_wrap']

NameError: name 'name2price' is not defined

## ⚠️ 5 Common Mistakes (Unpacking Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | A short arg-count on `f(*list)` (TypeError!) | The function's params = match the list-length, or have a defaults backup |
| 2 | A classic function rejecting `**dict`'s extra-keys — unexpected kwarg TypeError | put a `**kwargs` gobble-up inside the function |
| 3 | Assuming kwargs is the caller's own data | kwargs is Python's FRESH dict — the caller's vars are safe, play freely inside! |
| 4 | `f(a=1, *args, a=2)` duplicate kwarg — TypeError! `got multiple values` | dedupe the name-collisions |
| 5 | `**kwargs` before `*args` in the signature | ORDER law: pos → *args → kw-only → **kwargs (fixed!) |

> 🎯 **Library-design rule:** keep a `**kwargs` gobble-slot in public functions —
> tomorrow's new features get absorbed as keywords, old callers stay unbroken! 🧙 (a pro move)

## Summary

- `**kwargs` packs extra named arguments into a dictionary; `.get` reads optional fields safely.
- The full signature is positional, `*args`, keyword-only, `**kwargs`, in that fixed order.
- At the call site, `f(*list)` unpacks positionally and `f(**dict)` unpacks by name.
- A combined `packer(*args, **kwargs)` can count and forward both packs.
- `functools.partial` freezes some arguments to create a simpler callable.
- Applied to real data, these tools built an 18%-off discount lab and a dynamic receipt that ignores the flags it does not use.